In [2]:
import os
import random
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torchvision.models as models
from torch.utils.data import DataLoader, Subset
import pandas as pd

num_epochs = 300
batch_size = 32
learning_rate = 0.001

transform = transforms.Compose([
    transforms.RandomResizedCrop(224),  
    transforms.RandomHorizontalFlip(),    
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

data_dir = 'C:/Users/승승협/Desktop/Resnet18/data/800data'
full_dataset = datasets.ImageFolder(root=data_dir, transform=transform)

num_samples = len(full_dataset)
indices = list(range(num_samples))
random.shuffle(indices)

split = int(0.8 * num_samples)
train_indices, valid_indices = indices[:split], indices[split:]

train_dataset = Subset(full_dataset, train_indices)
valid_dataset = Subset(full_dataset, valid_indices)

train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(dataset=valid_dataset, batch_size=batch_size, shuffle=False)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def train_and_validate(model, train_loader, valid_loader, criterion, optimizer, num_epochs, result_filename):
    train_losses = []  
    valid_losses = []  
    train_accuracies = []  
    valid_accuracies = []  

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        train_correct = 0
        total_train = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_train += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        train_accuracy = train_correct / total_train
        train_loss /= len(train_loader)

        
        model.eval()
        valid_loss = 0.0
        valid_correct = 0
        total_valid = 0

        with torch.no_grad():
            for images, labels in valid_loader:
                images, labels = images.to(device), labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                valid_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total_valid += labels.size(0)
                valid_correct += (predicted == labels).sum().item()

        valid_accuracy = valid_correct / total_valid
        valid_loss /= len(valid_loader)

        
        print(f'Epoch [{epoch+1}/{num_epochs}], '
              f'Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.4f}, '
              f'Valid Loss: {valid_loss:.4f}, Valid Accuracy: {valid_accuracy:.4f}')
        
        
        train_losses.append(train_loss)
        valid_losses.append(valid_loss)
        train_accuracies.append(train_accuracy)
        valid_accuracies.append(valid_accuracy)

        
        if valid_accuracy > 0.8:  
            print("Validation accuracy exceeded 0.8, stopping training.")
            break  

    
    results_df = pd.DataFrame({
        'Epoch': range(1, len(train_losses) + 1),
        'Train Loss': train_losses,
        'Valid Loss': valid_losses,
        'Train Accuracy': train_accuracies,
        'Valid Accuracy': valid_accuracies
    })
    results_df.to_csv(result_filename, index=False)

    torch.save(model.state_dict(), result_filename.replace('.csv', '.pth'))

# Dropout rate 테스트
for dropout_rate in [0, 0.2, 0.3, 0.5]:
    print(f"Testing with dropout rate: {dropout_rate}")
    
    model = models.resnet18(pretrained=True)  
    num_ftrs = model.fc.in_features

    model.fc = nn.Sequential(
        nn.Linear(num_ftrs, num_ftrs),
        nn.ReLU(),
        nn.Dropout(dropout_rate),
        nn.Linear(num_ftrs, len(full_dataset.classes))
    )

    model.to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    result_filename = f'resnet18_epoch300_dropout{int(dropout_rate * 10)}_results.csv'
    train_and_validate(model, train_loader, valid_loader, criterion, optimizer, num_epochs, result_filename)


FileNotFoundError: [WinError 3] 지정된 경로를 찾을 수 없습니다: 'C:/Users/승승협/Desktop/Resnet18/data/800data'